In [ ]:
ReportFolderName = 'BERT-Based_Base_Version'

In [ ]:
import shutil
import os

# Keywords for folders to delete
folders_to_delete = ["logs", ReportFolderName, "results", "sample_data"]

# Delete matching folders
for item in os.listdir("."):
    if os.path.isdir(item) and any(keyword in item for keyword in folders_to_delete):
        shutil.rmtree(item)
        print(f"✅ Deleted folder: {item}")

# Delete all files in the current directory
for item in os.listdir("."):
    if os.path.isfile(item):
        os.remove(item)
        print(f"🗑️ Deleted file: {item}")

print("\n🎯 Full cleanup completed. All matching folders and all files removed.")


🎯 Full cleanup completed. All matching folders and all files removed.


In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

!pip install -q --upgrade transformers datasets peft accelerate scikit-learn tqdm

import torch, string, numpy as np, pandas as pd
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

In [ ]:
random_state = 43

In [ ]:
import re
def pre_process_sms(text):
    text = re.sub(r"http\S+", "URL", text)
    # text = re.sub(r"\d+", "NUM", text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

def preprocess_text(batch):
    batch["text"] = pre_process_sms(batch["text"])
    return batch

In [ ]:
label2id = {"normal": 0, "promo": 1, "smish": 2}
id2label = {v: k for k, v in label2id.items()}

def encode_labels(batch):
    batch["label"] = label2id[batch["label"]]
    return batch

In [ ]:
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

In [ ]:
# Only for two variant
variants = ["Bengali", "English"]
train_dataset_filtered = dataset["train"].filter(lambda example: example["source"] in variants)
dataset = DatasetDict({
    "train": train_dataset_filtered,
    "validation": dataset["validation"],
    "test": dataset["test"]
})


In [ ]:
dataset = dataset.map(encode_labels)
dataset = dataset.map(preprocess_text)

In [ ]:
dataset['train'][2]

{'label': 1,
 'text': '20 discount on travel packages contact today for booking',
 'source': 'English'}

In [ ]:
train_dataset = dataset['train']
val_dataset = dataset['validation']
test_dataset = dataset['test']

In [ ]:
train_df = pd.DataFrame(train_dataset)
print(train_df["source"].value_counts())

source
Bengali    1432
English    1388
Name: count, dtype: int64


In [ ]:
test_df = pd.DataFrame(test_dataset)
print(test_df["source"].value_counts())

source
Banglish    360
Bengali     358
English     347
CodeMix     336
Name: count, dtype: int64


In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize


def evaluate_and_report(trainer, test_dataset, label2id, model_alias, include_source=False):
    """
    Generates and saves evaluation reports for a trained model:
      - Classification report (text, CSV, LaTeX)
      - Confusion matrix (overall + per-source)
      - ROC curve (multi-class one-vs-rest)
      - Appends model summary to ./reports/summary.csv

    All outputs saved directly in ./reports/ (no per-model subfolders)
    """

    print(f"\n📊 Generating Evaluation Report for {model_alias}")

    # -----------------------------
    # 1️⃣ Prepare directory
    # -----------------------------
    report_dir = f"./{ReportFolderName}"
    os.makedirs(report_dir, exist_ok=True)

    # -----------------------------
    # 2️⃣ Predictions
    # -----------------------------
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]

    np.save(f"{report_dir}/{model_alias}_y_true.npy", y_true)
    np.save(f"{report_dir}/{model_alias}_y_pred.npy", y_pred)

    class_names = list(label2id.keys())

    # -----------------------------
    # 3️⃣ Classification Report
    # -----------------------------
    report_text = classification_report(y_true, y_pred, target_names=class_names)
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    # Text file
    with open(f"{report_dir}/{model_alias}_classification_report.txt", "w") as f:
        f.write(report_text)

    # CSV file
    df_report = pd.DataFrame(report_dict).transpose().round(4)
    df_report.to_csv(f"{report_dir}/{model_alias}_classification_report.csv")

    print(report_text)

    # -----------------------------
    # 4️⃣ Confusion Matrix
    # -----------------------------
    cm = confusion_matrix(y_true, y_pred, labels=list(label2id.values()))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {model_alias}")
    plt.tight_layout()
    plt.savefig(f"{report_dir}/{model_alias}_confusion_matrix.png")
    plt.close()

    # -----------------------------
    # 5️⃣ ROC Curve (One-vs-Rest)
    # -----------------------------
    try:
        y_true_bin = label_binarize(y_true, classes=list(label2id.values()))
        y_score = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()

        plt.figure(figsize=(6, 5))
        for i, class_name in enumerate(class_names):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f"{class_name} (AUC = {roc_auc:.2f})")

        plt.plot([0, 1], [0, 1], "k--", label="Random")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve - {model_alias}")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f"{report_dir}/{model_alias}_roc_curve.png")
        plt.close()
    except Exception as e:
        print(f"⚠️ Skipping ROC curve for {model_alias}: {e}")

    # -----------------------------
    # 6️⃣ Per-Source Evaluation (Confusion Matrix + Classification Report)

    if include_source and "source" in test_dataset.column_names:
        sources = test_dataset["source"]
        all_source_reports = []  # store metrics for summary

        for src in set(sources):
            mask = [s == src for s in sources]
            y_true_src = np.array(y_true)[mask]
            y_pred_src = np.array(y_pred)[mask]

            cm_src = confusion_matrix(y_true_src, y_pred_src, labels=list(label2id.values()))
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm_src, annot=True, fmt="d", cmap="Blues",
                        xticklabels=class_names, yticklabels=class_names)
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title(f"Confusion Matrix - {model_alias} ({src})")
            plt.tight_layout()
            plt.savefig(f"{report_dir}/{model_alias}_confusion_matrix_{src}.png")
            plt.close()

            # --- Classification Report ---
            report_dict = classification_report(
                y_true_src, y_pred_src,
                labels=list(label2id.values()),
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )
            report_df = pd.DataFrame(report_dict).transpose().round(4)
            report_df.to_csv(f"{report_dir}/{model_alias}_classification_report_{src}.csv", index=True)

            # Add macro averages for summary
            all_source_reports.append({
                "source": src,
                "precision": round(report_dict["macro avg"]["precision"], 4),
                "recall": round(report_dict["macro avg"]["recall"], 4),
                "f1_score": round(report_dict["macro avg"]["f1-score"], 4)
        })


        # --- Summary Report Across Sources ---
        summary_df = pd.DataFrame(all_source_reports)
        summary_df.loc["Average"] = summary_df.mean(numeric_only=True)
        summary_df.to_csv(f"{report_dir}/{model_alias}_source_summary_report.csv", index=False)

        print("\n✅ Per-source classification reports saved.")
        print(f"✅ Summary report saved to: {report_dir}/{model_alias}_source_summary_report.csv")


    # -----------------------------
    # 7️⃣ Summary CSV (append)
    # -----------------------------
    acc = round(report_dict["accuracy"], 4)
    precision = round(report_dict["weighted avg"]["precision"], 4)
    recall = round(report_dict["weighted avg"]["recall"], 4)
    f1 = round(report_dict["weighted avg"]["f1-score"], 4)

    summary_dict = {
        "model": model_alias,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

    summary_path = os.path.join(report_dir, "summary.csv")
    if os.path.exists(summary_path):
        existing = pd.read_csv(summary_path)
        existing = pd.concat([existing, pd.DataFrame([summary_dict])], ignore_index=True)
        existing.to_csv(summary_path, index=False)
    else:
        pd.DataFrame([summary_dict]).to_csv(summary_path, index=False)

    print(f"\n✅ {model_alias} → Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(f"✅ All reports saved in {report_dir}")

    return summary_dict


In [ ]:
def tokenize(batch):
    tokenized = tokenizer(batch["text"], truncation=True, max_length=128, padding="max_length")
    tokenized["label"] = batch["label"]
    return tokenized


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
reports_dir = f".{ReportFolderName}"
os.makedirs(reports_dir, exist_ok=True)

In [ ]:
# ============================================
# 2️⃣ DEFINE BASE MODELS
# ============================================

base_models = {
    "mBERT": "bert-base-multilingual-cased",
    "XLM-RoBERTa": "xlm-roberta-base",
    # 'XLM-RoBERTa Large': 'facebook/xlm-roberta-xl',
    # 'Muril': 'google/muril-base-cased',
    'Muril': 'google/muril-large-cased',
    'Distil-mBERT': 'distilbert-base-multilingual-cased',
}

meta_train_features = []
meta_test_features = []
all_model_results = []

# ============================================
# 3️⃣ LOOP THROUGH EACH BASE MODEL
# ============================================

for model_alias, model_name in base_models.items():
    print(f"\n🔥 Fine-tuning Base Model: {model_alias}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset =  train_dataset.map(tokenize, batched=True)

    val_dataset   = val_dataset.map(tokenize, batched=True)
    test_dataset  = test_dataset.map(tokenize, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id)
    target_modules=["query", "key", "value", "dense"]
    if model_name == 'distilbert-base-multilingual-cased':
      target_modules=["attention.q_lin", "attention.k_lin", "attention.v_lin", "attention.out_lin"]  # DistilBERT names
    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS"
    )
    print(target_modules)
    model = get_peft_model(base_model, lora_config)

    training_args = TrainingArguments(
    output_dir=f"./results_{model_alias}",
    learning_rate =  3e-5, #2e-5
    per_device_train_batch_size=32, # 16, 32, 64
    per_device_eval_batch_size=32, # 16, 32, 64
    num_train_epochs=10,
    weight_decay=0.01,
    # max_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=True
)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        seed=43,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Collect softmax probabilities for stacking
    preds_train = trainer.predict(train_dataset)
    preds_test  = trainer.predict(test_dataset)

    meta_train_features.append(torch.softmax(torch.tensor(preds_train.predictions), dim=1).numpy())
    meta_test_features.append(torch.softmax(torch.tensor(preds_test.predictions), dim=1).numpy())

    # Evaluate model and save reports
    metrics = evaluate_and_report(trainer, test_dataset, label2id, model_alias, include_source=True)
    all_model_results.append(metrics)




🔥 Fine-tuning Base Model: mBERT


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

A ConfigError was raised whilst setting the number of model parameters in Weights & Biases config.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.726891,0.678571,0.546512,0.678571,0.592411
2,No log,0.521314,0.804286,0.819716,0.804286,0.799973
3,No log,0.415617,0.831429,0.843783,0.831429,0.828332
4,No log,0.381106,0.858571,0.866765,0.858571,0.859094
5,No log,0.350678,0.865714,0.872580,0.865714,0.865802
6,0.476300,0.341136,0.874286,0.881209,0.874286,0.875262
7,0.476300,0.330911,0.884286,0.890456,0.884286,0.885094
8,0.476300,0.325927,0.888571,0.893855,0.888571,0.889171
9,0.476300,0.317725,0.888571,0.893706,0.888571,0.889238
10,0.476300,0.317462,0.888571,0.893706,0.888571,0.889238


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



📊 Generating Evaluation Report for Distil-mBERT


              precision    recall  f1-score   support

      normal       0.97      0.89      0.93       498
       promo       0.83      0.83      0.83       342
       smish       0.86      0.93      0.89       561

    accuracy                           0.89      1401
   macro avg       0.89      0.88      0.88      1401
weighted avg       0.89      0.89      0.89      1401


✅ Per-source classification reports saved.
✅ Summary report saved to: ./TrainWith2Varients/Distil-mBERT_source_summary_report.csv

✅ Distil-mBERT → Accuracy: 0.9280, F1: 0.9281
✅ All reports saved in ./TrainWith2Varients


In [ ]:
import shutil
from google.colab import files
import os

# List of folders to zip and download
folders_to_download = [
    ReportFolderName,
    #"stacking_ensemble_reports"
]

for folder in folders_to_download:
    if os.path.exists(folder):
        zip_filename = f"{folder}.zip"
        # Create zip archive
        shutil.make_archive(folder, 'zip', folder)
        # Download zip
        files.download(zip_filename)
        print(f"✅ Download started for '{zip_filename}'")
    else:
        print(f"⚠️ Folder '{folder}' not found")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started for 'TrainWith2Varients.zip'
